In [1]:
import sys
# sys.path에 상위 폴더(..)를 추가합니다.
sys.path.append("..")

from src.voltaflow.connectors.db import DbConnector
import requests
import os
from pathlib import Path
from src.voltaflow.parsers.Factory import FactoryProcessor
from assets import config
from src.voltaflow.connectors.minio import MinioConnector
import botocore
import io
import time
import re

In [2]:
header_kr = { 
        "Accept" : "application/json, text/javascript, */*; q=0.01",
        "Accept-Encoding": "gzip, deflate",
        "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
        "Connection": "keep-alive",
        "Content-Length": "23",
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
        "Cookie": "L-VISITOR=z2qurp1uebcoa6; JSESSIONID=E61974443D46EC4B8BD1F3548679E727.6134f0ddc45e87161; _ga_W4K6YD6SKQ=GS1.1.1718675776.1.1.1718675789.0.0.0; _ga_W4K6YD6SKQG-W4K6YD6SKQ=GS1.1.1718675777.1.1.1718675789.0.0.0; _fbp=fb.1.1731974383088.868934801414755891; _ga=GA1.1.521480946.1681885360; _ga_QMD5NW261X=GS1.1.1738630325.29.1.1738630376.0.0.0; _ga_JMHFCNW1FS=GS1.1.1738630325.30.1.1738630376.9.0.0; InitechEamUURL=http%3A%2F%2Ftoscn.lgensol.com%3A8009%2Ftos%2Fmain.do; InitechEamRTOA=1; CLP=; InitechEamCookieEnc=T; LGCHEM_NC_FLAG=TRUE; ep_mode=F; InitechEamNCFlag=F; language=ko; change=Y; InitechEamUID=1mL53LU3fyaHIWvC6qPkMg%3D%3D; InitechEamUIP=umxQQq46C0M6UuJdGIkf%2Fw%3D%3D; InitechEamULAT=1742965988; InitechEamUTOA=1; InitechEamUPID=XQDvTAgJ4afUQUwvgr1wJA%3D%3D; InitechEamUHMAC=q9%2BMFjtELSpwb7T7TTuROLpJGNCLwpqbvlPtr4NTt6g%3D; eWLanguage=ko-KR; engpuid=oHMNDv1VP83vQ91gJ4P1xNulp9ocriXZHmThkwuCDXs=; tempLGDPID=hyukjun_jo; InitechEamDomain=LGENSOL; LENA-UID=bcc9c6d8.6316178c7fee8; SCOUTER=z2bd0rldnhe3l8; leftMenuType=list",
        "Host": "toskr.lgensol.com:8009",
        "Origin": "http://toskr.lgensol.com:8009",
        "Referer""": "http://toskr.lgensol.com:8009/tos/test/resultCdms/retrieveTestResultFileList.pop",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
        "X-Requested-With": "XMLHttpRequest"
}

header_cn = { 
        "Accept" : "application/json, text/javascript, */*; q=0.01",
        "Accept-Encoding": "gzip, deflate",
        "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
        "Connection": "keep-alive",
        "Content-Length": "23",
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
        "Cookie": "L-VISITOR=z2qurp1uebcoa6; JSESSIONID=E61974443D46EC4B8BD1F3548679E727.6134f0ddc45e87161; _ga_W4K6YD6SKQ=GS1.1.1718675776.1.1.1718675789.0.0.0; _ga_W4K6YD6SKQG-W4K6YD6SKQ=GS1.1.1718675777.1.1.1718675789.0.0.0; _fbp=fb.1.1731974383088.868934801414755891; _ga=GA1.1.521480946.1681885360; _ga_QMD5NW261X=GS1.1.1738630325.29.1.1738630376.0.0.0; _ga_JMHFCNW1FS=GS1.1.1738630325.30.1.1738630376.9.0.0; InitechEamUURL=http%3A%2F%2Ftoscn.lgensol.com%3A8009%2Ftos%2Fmain.do; InitechEamRTOA=1; CLP=; InitechEamCookieEnc=T; LGCHEM_NC_FLAG=TRUE; ep_mode=F; InitechEamNCFlag=F; language=ko; change=Y; InitechEamUID=1mL53LU3fyaHIWvC6qPkMg%3D%3D; InitechEamUIP=umxQQq46C0M6UuJdGIkf%2Fw%3D%3D; InitechEamULAT=1742965988; InitechEamUTOA=1; InitechEamUPID=XQDvTAgJ4afUQUwvgr1wJA%3D%3D; InitechEamUHMAC=q9%2BMFjtELSpwb7T7TTuROLpJGNCLwpqbvlPtr4NTt6g%3D; eWLanguage=ko-KR; engpuid=oHMNDv1VP83vQ91gJ4P1xNulp9ocriXZHmThkwuCDXs=; tempLGDPID=hyukjun_jo; InitechEamDomain=LGENSOL; LENA-UID=bcc9c6d8.6316178c7fee8; SCOUTER=z2bd0rldnhe3l8; leftMenuType=list",
        "Host": "toscn.lgensol.com:8009",
        "Origin": "http://toscn.lgensol.com:8009",
        "Referer""": "http://toscn.lgensol.com:8009/tos/test/resultCdms/retrieveTestResultFileList.pop",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
        "X-Requested-With": "XMLHttpRequest"
}

In [2]:
host = os.getenv("DB_HOST")
database = "mart_db"
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
port = os.getenv("DB_PORT")

db_conn = DbConnector(host, database, user, password, port)
db_conn.connect()
DESTINATION_TABLE_NAME = "pipeline_queue_tb"
query = f"SELECT * FROM {DESTINATION_TABLE_NAME} WHERE download_required_yn IS NULL OR TRUE;"
pipeline_queue_tb = db_conn.fetch_data(query ,as_dataframe=True)

데이터베이스 'mart_db'에 성공적으로 연결되었습니다.


In [42]:
res.status_code == 200

True

In [13]:
for idx, row in pipeline_queue_tb.iterrows():
    payload = {"analyzerUrl": row["outter_ip"],
                "testId":row["exp_id"]}
    print(row["test_title"])
    if row['test_site'] == '대전' or row['test_site'] == 'EP2':
        header = header_kr
        url = "http://toskr.lgensol.com:8009/tos/test/resultCdms/requestTestResultData.ajax"
    else:
        header = header_cn 
        url = "http://toscn.lgensol.com:8009/tos/test/resultCdms/requestTestResultData.ajax"
        
    res = requests.post(url,
                    headers=header,
                    data=payload)
    time.sleep(2)

20240507_JF1_MP_SII_JIS_45_0.5CCCV_0.5CC_DA05B19192
20250213_JF1_MP_SII_JIS_45_0.5CCCV_0.5CC_DA05B19192_re
250702_JF1_MP_SII_JIS_45_0.5CCCV_0.5CC_DA05B19192
20250214_JF1_MP_SII_JIS_45_0.5CCCV_0.5CC_DA05B19192_re
20240806_JF1_MP_SII_JIS_45_0.5CCCV_0.5CC_DA05B19192
20250527_JF1_MP_45_0.75CP_SOC5~100_EA23135723
20250417_JF1_MP_45_0.75CP_SOC5~100_EA23135723
20250612_JF1R_P1_T50_CPC_05CP_SOC70~100_DK11EB2545
20250501_JF2S_CCV_T55_CPC_1_1_12CP_SOC0~100_UDKTF10115
20250327_JF2S_CV_ACC_StepCD_55_1_1_12CP_SOC0~100_UDKTF10115
20250501_JF2S_CCV_T55_CPC_1_1_12CP_SOC0~100_UDKTF10102
20250327_JF2S_CV_ACC_StepCD_55_1_1_12CP_SOC0~100_UDKTF10102
20250501_JF2S_CCV_T55_CPC_55_12_1_1CP_SOC0~100_UDKTF10108
20250324_JF2S_CV_ACC_StepCD_55_12_1_1CP_SOC0~100_UDKTF10108
20250501_JF2S_CCV_T55_CPC_55_12_1_1CP_SOC0~100_UDKTF10105
20250502_JF1_M1_T45_CPC_JF1_MP_45_075CP_SOC5~100_EA23135708
20250502_JF1_M1_T45_CPC_JF1_MP_45_025CP_SOC5~100_EA23135634
20250416_JF2S_CV_ACC_55_09045CP_1CP_UDKTF10021_SOCcut
20250414_JF2S

In [14]:
row["exp_id"]

'EXP_240711_00516'

In [ ]:
for idx, row in pipeline_queue_tb.iterrows():
    payload = {"testId":row["exp_id"]}
    print(row["test_title"])
    if row['test_site'] == '대전' or row['test_site'] == 'EP2':
        header = header_kr
        url = "http://toskr.lgensol.com:8009/tos/test/testResultDownload/retrieveTstRltDldDtlForm.do"
    else:
        header = header_cn 
        url = "http://toscn.lgensol.com:8009/tos/test/testResultDownload/retrieveTstRltFileList.ajax"
        
    res = requests.post(url,
                    headers=header,
                    data=payload)
    time.sleep(2)
    

20240507_JF1_MP_SII_JIS_45_0.5CCCV_0.5CC_DA05B19192


In [27]:
row["exp_id"]

'EXP_240704_01166'

In [5]:
payload = {"searchTestTitle": "UDTF10123"}

res = requests.post("http://toscn.lgensol.com:8009/tos/test/resultCdms/retrieveTestResultList.ajax",
                    headers=header_kr,
                    data=payload)

In [6]:
res.json()

{'total': 1,
 'records': 4,
 'pageSize': 15,
 'page': 1,
 'rows': [{'customRowSize': [10, 20, 50],
   'imSysMgrYn': 'N',
   'iamJobTesterLeaderYn': 'N',
   'autoCompleteSearchExceptStr': '',
   'pageSize': 10,
   'iamJobTesterYn': 'N',
   'rsltCode': 'FAIL',
   'scheduleJobEqYn': 'N',
   'autoCompleteSelectedStr': '',
   'autoCompleteYn': 'N',
   'languageCd': 'cn',
   'groupingView': False,
   'testFileLastUploadDatetime': '2026-01-10 02:32:20',
   'rsltMssg': '',
   'paging': True,
   'iamTosTestLeaderYn': 'N',
   'lastChnnEndDatetime': '2025-06-11 09:20:26.0',
   'manufacturerCd': 'EQT18',
   'mobileDevice': False,
   'testStartDatetime': '2024-12-30 11:06:41.0',
   'testStateNm': '완료',
   'page': 1,
   'sord': '',
   'workplaceNm': 'ESNB',
   'testNmEmpNo': '20241230',
   'multiselectYn': 'N',
   'testFullTitle': '20241230_JF2S_45oC_0.5CP 0.67CP_Cycle_UDTF10123',
   'testChannelCnt': 1,
   'jobCheckSkipYn': 'N',
   'iamTeamLeaderYn': 'N',
   'iamTosOperLeaderYn': 'N',
   'equipNm':

In [8]:
def request_test_results(site, sub ,payload):
    if sub == "retrieveTstRltFileList":
        base_url = "http://tos{}.lgensol.com:8009/tos/testResultDownload/{}.ajax"
    else: # 'retrieveTestResultList', 'requestTestResultData
        base_url = "http://tos{}.lgensol.com:8009/tos/test/resultCdms/{}.ajax"
    
    res = requests.post(base_url.format(site, sub),
                headers=header_kr if site == 'kr' else header_cn,
                data=payload,
                timeout=30)
    return res

In [10]:
test_id = "EXP_241226_00024"
site ='cn'
payload = {"testId": test_id}
res = request_test_results(site, "retrieveTstRltFileList", payload)
res.json()['rows']

[{'testFullTitle': '20241226_JF2_45oC_0.5CP 0.67CP_Cycle_UDTF10123',
  'iamTeamLeaderYn': 'N',
  'iamTosOperLeaderYn': 'N',
  'customRowSize': [10, 20, 50],
  'modifyDatetime': '2026-01-10 02:17:23',
  'imSysMgrYn': 'N',
  'records': 0,
  'iamJobTesterLeaderYn': 'N',
  'chdchEquipId': 'P_CHDCH_202300251',
  'autoCompleteSearchExceptStr': '',
  'orderBy': '',
  'pageSize': 10,
  'fileFullPath': 'http://10.61.10.54:8008/2024/2024/12/P_CHDCH_202300251/20241226_JF2_45oC_0.5CP 0.67CP_Cycle_UDTF10123/M05CH018[018]/JF2S 0.5CP_0.67CP long term cycle.pln',
  'registDatetime': '2024-12-30 11:00:15',
  'iamTosEngineerYn': 'N',
  'iamJobTesterYn': 'N',
  'fileNm': 'JF2S 0.5CP_0.67CP long term cycle.pln',
  'fileExtension': 'pln',
  'rsltCode': 'FAIL',
  'resultSize': 0,
  'autoCompleteSelectedStr': '',
  'workplaceCd': 'WP_CNB',
  'autoCompleteYn': 'N',
  'sidx': '',
  'rowSize': 10,
  'iamEquipMgrYn': 'N',
  'defaultRowSize': 15,
  'languageCd': 'cn',
  'groupingView': False,
  'rsltMssg': '',
  